# Qoro × PushQuantum Hackathon — 01 · Portfolio optimisation (QOBLIB 06)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QoroQuantum/pushquantum-hackathon-2026/blob/main/01_portfolio_optimisation.ipynb)

Decide, day by day, how many units of each stock to hold long and short, trading risk against returns
and costs. This notebook goes from QOBLIB's official model to a QUBO, sets a classical baseline, attacks
the QUBO with PCE and partitioned QAOA, and writes a solution file in QOBLIB's format.

**Your challenge: scale up.** Instances run from 3 assets × 2 days (58 binary variables) to 400 assets ×
15 days (about 36 000). Best-known values are published up to 50 assets; for 200 and 400 assets
there are none yet (section 1.8). Solve the largest instance you can, as close to the best-known value as
you can.

**Scoring** uses QOBLIB's official objective, not your QUBO's energy: `qt.PortfolioModel.score` is a
port of QOBLIB's checker. Treat it as the referee.

**Before you start,** work through notebook 00 (setup and QUBO 101).

---
## 0 · Setup

Run these cells first. Notebook 00 (Part 0) explains them: your API key and the install. The helper file
`qoblib_tools.py` is downloaded for you.

In [1]:
# Setup, explained in notebook 00 (Part 0). If this cell installs Divi, restart the session
# (Runtime → Restart session) and run the notebook from the top.
import os
from importlib.metadata import PackageNotFoundError, version

try:
    divi_version = version("qoro-divi")
except PackageNotFoundError:
    divi_version = None
if divi_version != "0.16.2":
    %pip install -q "qoro-divi[qubo-decompose,jupyter]==0.16.2"
    print("Installed Divi 0.16.2: restart the session and run from the top.")

try:  # Google Colab: copy the secret QORO_API_KEY into the environment
    from google.colab import userdata
    os.environ["QORO_API_KEY"] = userdata.get("QORO_API_KEY")
except ImportError:
    pass  # not in Colab: Divi reads .env or the environment by itself
except Exception as err:  # no secret yet, or notebook access switched off
    print(f"No QORO_API_KEY in Colab Secrets yet ({type(err).__name__}): see notebook 00, section 0.1.")

In [2]:
# The helper file qoblib_tools.py (instance loaders and official scorers) is downloaded from the
# hackathon repository if it is not next to this notebook yet.
import urllib.request

if not os.path.exists("qoblib_tools.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/QoroQuantum/pushquantum-hackathon-2026/main/qoblib_tools.py", "qoblib_tools.py")
import qoblib_tools as qt

missing = [name for name in [
    "PortfolioModel", "PortfolioQUBO", "load_portfolio_instance", "read_portfolio_solution",
    "fetch_portfolio_reference",
] if not hasattr(qt, name)]
if missing:
    raise ImportError(
        f"{os.path.relpath(qt.__file__)} is an old copy (no {', '.join(missing)}). Delete it, restart the "
        "session and run from the top: the current version is downloaded."
    )
print("qoblib_tools loaded from", os.path.relpath(qt.__file__))

qoblib_tools loaded from qoblib_tools.py


In [3]:
import time
import warnings

warnings.filterwarnings("ignore", category=SyntaxWarning)  # noisy docstrings in some dependencies

import dimod
import hybrid
import requests
from dwave.samplers import SimulatedAnnealingSampler
from IPython.display import display

from divi.backends import JobConfig, MaestroSimulator, QoroService
from divi.backends.characterization import characterize_and_validate
from divi.qprog.algorithms import PCE, PolyEncoding, SoftObjective
from divi.qprog.ensembles import BeamSearchStrategy, PartitioningProgramEnsemble
from divi.qprog.optimizers import MonteCarloOptimizer
from divi.qprog.problems import BinaryOptimizationProblem, CommunityDecomposer, QUBOPartitioningConfig

print("qoro-divi", version("qoro-divi"))

qoro-divi 0.16.1


In [4]:
USE_CLOUD = False            # False: local Maestro simulator · True: Qoro cloud (uses credits)
RUN_CHARACTERIZATION = True  # Characterization Service calls (cloud; cost scales with QUBO size)
SHOTS = 10_000
MC_POPULATION = 50          # Monte Carlo optimiser: circuits per iteration, sent to the backend as one batch
MC_ITERATIONS = 10          # Monte Carlo optimiser iterations

try:
    service = QoroService()  # reads QORO_API_KEY from .env or the environment
    balance = service.get_credit_balance()
    print("Connected to Qoro. Credit balance:", balance.get("balance", balance))
except (ValueError, requests.RequestException) as err:  # no key, a wrong key, or no connection
    service = None
    print(f"{err}\nNo working API key (notebook 00, section 0.1): cloud cells will be skipped.")
if USE_CLOUD and service is None:
    raise RuntimeError("USE_CLOUD = True needs a working API key: see notebook 00, section 0.1.")


def make_backend(shots: int = SHOTS, force_sampling: bool = False):
    # force_sampling=True: estimate everything from shots instead of exact expectation values
    if USE_CLOUD:
        return QoroService(
            job_config=JobConfig(
                shots=shots, simulator_cluster="qoro_maestro", force_sampling=force_sampling
            )
        )
    return MaestroSimulator(shots=shots, force_sampling=force_sampling)


backend = make_backend()
print("Circuits run on:", type(backend).__name__)

Connected to Qoro. Credit balance: <your balance>
Circuits run on: MaestroSimulator


**`solve_partitioned`** is the helper from notebook 00 (section 1.5), with one addition: an optional
`decomposer` argument, for when you want to choose the partitions yourself.

In [5]:
def solve_partitioned(bqm, routine="qaoa", cluster_size=12, n_layers=2, backend=backend, decomposer=None, **routine_kwargs):
    # Partition `bqm`, solve every part with `routine`, and return the ensemble.
    # routine: "qaoa" | "iterative_qaoa" | "pce" | "progressive_pce"
    problem = BinaryOptimizationProblem(
        bqm,
        decomposer=decomposer or CommunityDecomposer(
            QUBOPartitioningConfig(max_n_variables_per_cluster=cluster_size)
        ),
        composer=hybrid.SplatComposer(),
        local_search=True,
    )
    ensemble = PartitioningProgramEnsemble(
        problem=problem,
        quantum_routine=routine,
        n_layers=n_layers,
        optimizer=MonteCarloOptimizer(population_size=MC_POPULATION),
        max_iterations=MC_ITERATIONS,
        backend=backend,
        **routine_kwargs,
    )
    ensemble.run()
    return ensemble

---
## 1 · QOBLIB 06: multi-period portfolio optimisation

Over $T$ trading days, decide for each of $n$ stocks how many **units** to hold **long** and how many
**short** (0–3 each). One unit is worth 100 000 at day 0. The objective to minimise is

$$\underbrace{\lambda\,\text{risk}}_{\text{covariance}} \;-\; \text{returns} \;+\; \text{transaction costs} \;-\; \text{interest on idle cash} \;+\; \text{short fees} \;+\; \text{liquidation costs}$$

subject to two rules on every day:

* **Capital:** net position (long − short units) must lie between $C-15$ and $C = 10$.
* **Budget:** at most $B$ open units.

QOBLIB's model ([`bqp_u3_c10.zpl`](https://github.com/ZIB-AOPT/QOBLIB/tree/main/06-portfolio/models))
uses one binary variable per *(asset, unit copy, direction, day)* plus binary slack registers that
turn both rules into equalities. That gives about $6nT + 11T$ variables: **710 for 10 assets × 10 days**.

* **Instances:** `po_aXXX_tYY_sZZ` means XXX assets, YY days, data seed ZZ (or `orig`). Each is
  solved for 8 risk weights $\lambda \in \{0, 10^{-6}, \dots, 10^{-2}\}$ with $B$ = 4, 20, 50 or 100
  for 10, 50, 200 or 400 assets.
* **Score:** `PortfolioModel.score(positions)` computes QOBLIB's objective from your **unit counts
  only**. Use any QUBO you like, as long as you decode it to unit counts.

### 1.1 A tiny instance and the referee

In [6]:
inst = qt.load_portfolio_instance("po_a003_t02_orig")  # 3 assets × 2 days (downloaded once)
model = qt.PortfolioModel(inst, budget=3, lam="1e-5")

print("Value of one unit over time:")
for i, sym in enumerate(inst.symbols):
    print(f"  {sym:>5}: {[round(float(p)) for p in model.unit_prices[i]]}")

Value of one unit over time:
   AAPL: [100000, 99154]
   NVDA: [100000, 100008]
   MSFT: [100000, 100915]


First, check the referee against a published QOBLIB result. Then compute the exact optimum of the tiny
instance by dynamic programming. This only works for about 3 assets, but it gives a ground truth to test
your pipeline against.

In [7]:
ref_model = qt.PortfolioModel(qt.load_portfolio_instance("po_a010_t10_orig"), budget=4, lam="1e-4")
header, ref_positions = qt.read_portfolio_solution(
    qt.fetch_portfolio_reference("po_a010_t10_orig", budget=4, lam_tag="1e-4")
)
print("QOBLIB reference solution — published objective:", header["objective"])
print("                              our referee says:   ", ref_model.score(ref_positions))

best_positions, best_obj = model.brute_force()
print("\nTiny instance — exact optimum:", best_obj)
print("  positions {(day, symbol): (long, short)}:", best_positions)
print("  ", model.score(best_positions))

QOBLIB reference solution — published objective: -69482
                              our referee says:    objective=-69482 (feasible; risk=23416, return=-97869, transaction=5821, cash_interest=-880, short_cost=30, liquidation=0)

Tiny instance — exact optimum: -2505
  positions {(day, symbol): (long, short)}: {(0, 'AAPL'): (0, 1), (0, 'MSFT'): (2, 0)}
   objective=-2505 (feasible; risk=58, return=-2676, transaction=300, cash_interest=-190, short_cost=3, liquidation=0)


### 1.2 From model to QUBO

`PortfolioQUBO` builds QOBLIB's model as two binary quadratic models over the same variables:

* `objective`: the cost function.
* `penalty`: the two squared equality constraints, zero exactly on feasible bitstrings.

The solver sees `objective + W · penalty`, and **the penalty weight `W` is your most important knob.**
If `W` is too small, samples break the rules. If it's too large, the objective is drowned out and
quality suffers.

Keyword arguments let you change the encoding while keeping the same score. Fewer copies (`ub`) or
narrower slack registers mean fewer variables.

In [8]:
q = qt.PortfolioQUBO(model)
print(f"{q.n_vars} binary variables = {q.n_x} position bits + {q.n_vars - q.n_x} slack bits")

bits = q.encode(best_positions)  # positions → bits (unit copies + exact slack values)
print("objective energy of the optimum:", q.objective.energy(dict(enumerate(bits))),
      "| penalty:", q.penalty.energy(dict(enumerate(bits))))


def penalised(q: qt.PortfolioQUBO, weight: float, normalize: bool = False) -> dimod.BinaryQuadraticModel:
    # objective + weight·penalty as one BQM (optionally rescaled so |coefficients| ≤ 1).
    bqm = q.objective.copy()
    penalty = q.penalty.copy()
    penalty.scale(weight)
    bqm.update(penalty)
    if normalize:
        bqm.normalize()
    return bqm


def best_polished(model: qt.PortfolioModel, q: qt.PortfolioQUBO, samples, top: int = 10):
    # Decode the first `top` samples, repair + local-search each, return (positions, score) of the best.
    candidates = [model.local_search(q.decode(s)) for s in list(samples)[:top]]
    best = min(candidates, key=lambda p: model.score(p).objective)
    return best, model.score(best)

58 binary variables = 36 position bits + 22 slack bits
objective energy of the optimum: -2505.0 | penalty: 0.0


### 1.3 Characterize the portfolio QUBO and tune the penalty

Pass the objective and the penalty *separately*, using `BinaryOptimizationProblem(objective, penalty=…)`.
Divi then sends the cost/penalty split on its own, and the report gains a `penalty` section with the
weight thresholds and, for small enough problems, a recommended weight (`recommended_lambda`).

The `structure` section is worth reading too. It shows how well the QUBO splits into parts, which
helps you choose a partition size in 1.6.

Divi warns that the QUBO's constant offset isn't sent: the reported energies exclude it, and nothing else changes.

In [9]:
W = 1000.0  # starting guess

pf_report = None
if service is not None and RUN_CHARACTERIZATION:
    pf_report = characterize_and_validate(
        BinaryOptimizationProblem(q.objective, penalty=q.penalty, penalty_weight=W),
        reference_states=[q.bitstring(best_positions)],
        service=service,
    )
    display(pf_report)
    print("penalty section:   ", pf_report.penalty)
    print("structure section: ", pf_report.structure)
    if pf_report.recommended_lambda:
        W = float(pf_report.recommended_lambda)
else:
    print("Skipped: no API key, or RUN_CHARACTERIZATION is False.")
print("Using penalty weight W =", W)

/tmp/ipykernel_17176/511804533.py:5: UserWarning: The problem carries a constant offset, which is not transmitted. The analysis is unaffected — a constant shifts every energy equally — but the reported energies exclude it.
  pf_report = characterize_and_validate(


CharacterizationResult(job_id='b421fcf4-b1a2-48f7-8096-e0baafcc63f5', status='COMPLETED')
  QAOA was still improving at p = 1 layer, the deepest computed: use at least that many.
  [action] Fixing variables 50, 51, 52, 53, 54, 55, 56, 57 at the values listed under structure keeps an optimum: substitute them and the problem left needs 50 qubits.
  [action] The problem splits into 2 independent parts; solve them separately.
  [info] The best classical solution is within 0.1% of a rigorous lower bound (semidefinite-relaxation bound after fixing variables). That is the most any solver can still gain; the bound itself can sit below the true minimum.
  [info] The best solution came from rounding the semidefinite relaxation; 3 of 136 local searches reached it, so about 207 independent runs find it with 99% confidence. 9 of 50 variables take the same value in every near-best solution.
  [info] Splitting what is left into 4 parts of at most 13 variables cuts 40.8% of its coupling weight: solving the parts separately is off by at most 1.025e+06 in energy. Coarser splits are listed under structure.
  [info] At the requested p = 1 layer QAOA's mean energy closes between 81.3% and 81.3% of the gap from random guessing to the optimum.
  [info] Layer counts past 1 were not computed: every light-cone is wider than 22 qubits.
  [info] λ > 2262 keeps the ground state feasible if every violation adds at least 1 to the penalty; a smaller λ keeps the cost landscape sharper.

penalty section:    {'exact': False, 'qaoa_lambda': 1000.0, 'safe_lambda': 2262.0, 'penalty_to_cost': 1374.5888814381965, 'assumes_unit_violation': True}
structure section:  {'fixed_by': {'first_order': 8, 'roof_duality': 0}, 'partition': [{'parts': 4, 'cut_share': 0.40813963673427134, 'cut_weight': 1025108.0, 'largest_part': 13}], 'components': 2, 'fixed_to_one': [], 'z2_symmetric': False, 'fixed_to_zero': [50, 51, 52, 53, 54, 55, 56, 57], 'easy_witnesses': [], 'penalty_blocks': [], 'unused_variables': [], 'largest_component': 29, 'qubits_after_reduction': 50}
Using penalty weight W = 1000.0


### 1.4 Classical baseline: simulated annealing → repair → local search

**Always know your classical bar.** `model.repair` makes a solution feasible by dropping the least
valuable units, and `model.local_search` then applies single-unit moves while they improve the
*official* score. Use both as post-processing on any sample, quantum or classical.

In [10]:
t0 = time.time()
sa = SimulatedAnnealingSampler().sample(penalised(q, W), num_reads=100, num_sweeps=1000, seed=1)
n_feasible = sum(model.score(q.decode(s)).feasible for s in sa.samples())
sa_positions, sa_score = best_polished(model, q, sa.samples())
print(f"SA: {n_feasible}/100 raw samples feasible; after repair + local search: {sa_score.objective} "
      f"({time.time() - t0:.1f}s) — optimum {best_obj}")

SA: 73/100 raw samples feasible; after repair + local search: -2505 (0.1s) — optimum -2505


### 1.5 Quantum route A: PCE on the whole QUBO

58 variables won't fit in one QAOA circuit on a laptop, but `PolyEncoding` needs only 11 qubits.

In [11]:
pf_problem = BinaryOptimizationProblem(penalised(q, W, normalize=True))
t0 = time.time()
pce_pf = PCE(
    pf_problem,
    encoding=PolyEncoding(),
    objective=SoftObjective(alpha=2.0),
    n_layers=3,
    optimizer=MonteCarloOptimizer(population_size=MC_POPULATION),
    max_iterations=MC_ITERATIONS,
    backend=backend,
    seed=1,
)
pce_pf.run()
top = pce_pf.get_top_solutions(n=20, min_prob=0.0, sort_by="energy")
raw_best = min(model.score(q.decode(s.bitstring)).objective for s in top)
pce_positions, pce_score = best_polished(model, q, [s.bitstring for s in top])
print(f"PCE: {q.n_vars} variables on {pce_pf.n_qubits} qubits, {pce_pf.total_circuit_count} circuits, "
      f"{time.time() - t0:.1f}s")
print(f"  best raw sample: {raw_best} | after repair + local search: {pce_score.objective} (optimum {best_obj})")

PCE: 58 variables on 11 qubits, 501 circuits, 0.2s


  best raw sample: -741 | after repair + local search: -2505 (optimum -2505)


### 1.6 Quantum route B: partition → QAOA → stitch → polish

In [12]:
pf_bqm = penalised(q, W, normalize=True)
t0 = time.time()
ensemble = solve_partitioned(pf_bqm, "qaoa", cluster_size=10)
tops = ensemble.get_top_solutions(n=10, strategy=BeamSearchStrategy(beam_width=10, n_partition_candidates=5))
samples = [dict(zip(pf_bqm.variables, solution)) for solution, _ in tops]
part_positions, part_score = best_polished(model, q, samples)
print(f"Partitioned QAOA: {len(ensemble.programs)} sub-programs, {ensemble.total_circuit_count} circuits, "
      f"{time.time() - t0:.1f}s → {part_score.objective} after polishing (optimum {best_obj})")

Partitioned QAOA: 8 sub-programs, 4008 circuits, 1.4s → -2505 after polishing (optimum -2505)


### 1.7 Scale up to a leaderboard instance

`po_a010_t10_orig` with $B = 4$ and $\lambda = 10^{-4}$: 710 variables, best-known value **−69 482**.
With `USE_CLOUD = True` the 70 sub-programs of each run execute in parallel on Qoro's cloud.
Each run sends 70 × 501 ≈ 35 000 circuits (lower `MC_POPULATION` or `MC_ITERATIONS` to
spend fewer credits). Locally, each run takes under a minute, depending on your machine.

Results vary from run to run, so we try three seeds and keep the best. When you're hunting a record,
running many attempts and keeping the best is fair play.

In [13]:
INSTANCE, BUDGET, LAM, BKV = "po_a010_t10_orig", 4, "1e-4", -69482
W_BIG = 100.0  # smaller weights let the objective steer the search; repair restores feasibility

big_model = qt.PortfolioModel(qt.load_portfolio_instance(INSTANCE), budget=BUDGET, lam=LAM)
big_q = qt.PortfolioQUBO(big_model)
print(f"{INSTANCE}, B={BUDGET}, λ={LAM}: {big_q.n_vars} QUBO variables")
print("Local search from an empty portfolio:", big_model.score(big_model.local_search({})).objective)

big_bqm = penalised(big_q, W_BIG, normalize=True)
runs = []
for seed in (1, 2, 3):
    t0 = time.time()
    big_ensemble = solve_partitioned(big_bqm, "qaoa", cluster_size=12, backend=make_backend(), seed=seed)
    big_tops = big_ensemble.get_top_solutions(
        n=10, strategy=BeamSearchStrategy(beam_width=10, n_partition_candidates=5)
    )
    big_samples = [dict(zip(big_bqm.variables, solution)) for solution, _ in big_tops]
    positions, score = best_polished(big_model, big_q, big_samples)
    runs.append((score.objective, positions))
    print(f"seed {seed}: {score.objective} ({len(big_ensemble.programs)} sub-programs, "
          f"{big_ensemble.total_circuit_count} circuits, {time.time() - t0:.0f}s)")

best_run, big_positions = min(runs, key=lambda r: r[0])
big_score = big_model.score(big_positions)
print(f"\nBest of {len(runs)} runs: {best_run} — BKV {BKV}, gap {(best_run - BKV) / abs(BKV):+.2%}")
print(big_score)

po_a010_t10_orig, B=4, λ=1e-4: 710 QUBO variables
Local search from an empty portfolio: -69372


seed 1: -69169 (70 sub-programs, 35070 circuits, 25s)


seed 2: -68949 (70 sub-programs, 35070 circuits, 26s)


seed 3: -69029 (70 sub-programs, 35070 circuits, 45s)

Best of 3 runs: -69169 — BKV -69482, gap +0.45%
objective=-69169 (feasible; risk=26483, return=-101020, transaction=6015, cash_interest=-880, short_cost=33, liquidation=200)


Write your solution in QOBLIB's canonical `.sol` format. This is the file you hand in.

In [14]:
lam_tag = LAM.replace("-0", "-")
sol_path = f"{INSTANCE}_b{BUDGET:03d}_l{lam_tag}.sol"
result = big_model.write_solution(sol_path, big_positions, comment="Qoro x QOBLIB hackathon: partitioned QAOA + local search")
print(f"Wrote {sol_path}: {result}")
print(open(sol_path).read()[:400])

Wrote po_a010_t10_orig_b004_l1e-4.sol: objective=-69169 (feasible; risk=26483, return=-101020, transaction=6015, cash_interest=-880, short_cost=33, liquidation=200)
# Qoro x QOBLIB hackathon: partitioned QAOA + local search
instance po_a010_t10_orig
budget 4
lambda 0.0001
objective -69169
# period symbol long short
0 AAPL 0 1
0 GOOGL 3 0
1 META 1 0
1 TSLA 0 3
2 META 3 0
2 NVDA 0 1
3 AVGO 0 1
3 GOOG 0 2
3 META 1 0
4 AMZN 1 0
4 META 2 0
4 TSLA 0 1
5 META 0 1
5 TSLA 3 0
6 GOOG 1 0
6 MSFT 1 0
6 NVDA 0 1
6 TSLA 1 0
7 TSLA 0 1
7 WMT 3 0
8 AAPL 3 0
8 AVGO 1 0
9 NVDA


### 1.8 Targets and ideas

Best-known values from the [QOBLIB table](https://github.com/ZIB-AOPT/QOBLIB/blob/main/06-portfolio/solutions/README.md)
(check it for updates), with what this notebook's classical starter tools reach. "SA + LS" is
simulated annealing on the official QUBO (W = 100), followed by `local_search`.

| Instance | B | λ | Best known | Starter (SA + LS) | |
|---|---:|---:|---:|---:|---|
| `po_a010_t10_orig` | 4 | 1e-4 | −69 482 | −69 482 | matched: warm-up |
| `po_a010_t10_orig` | 4 | 5e-4 | −27 044 | −26 548 | gap 1.8 % |
| `po_a010_t10_s02`  | 4 | 1e-4 | −53 255 | −51 805 | gap 2.7 % |
| `po_a050_t10_orig` | 20 | 1e-3 | −206 239 | −197 953 | gap 4.0 % |
| `po_a010_t10_orig` | 4 | 1e-3 | −8 397 | −3 703 | gap 56 % |
| `po_a050_t10_orig` | 20 | 1e-2 | −37 107 | −1 000 (local search alone) | gap 97 % |
| `po_a200_*`, `po_a400_*` | 50 / 100 | any | *none published* | — | uncharted |

The large-λ instances are where risk matters. Profitable portfolios need *hedged* long/short pairs,
which is a strongly coupled, frustrated QUBO. Single-unit moves get stuck there.

**Ideas to try**
* **Penalties.** Sweep `W`, use the characterizer's recommendation, or replace equality + slack with
  *unbalanced penalisation* of the inequalities. Feasibility is scored, not your penalty.
* **Encoding.** Use fewer copies (`ub=2`), narrower slack registers (`count_slack_bits=3` covers
  B = 4), or integer (binary) encoding of unit counts instead of unary copies. The cash interest
  equals $-10\,(C - \text{net position})$, which is linear in your positions. Fold it into the
  objective and enforce the capital rule as an inequality (for example, unbalanced penalisation) to
  drop the cash slack bits.
* **Partitioning.** Day-by-day blocks are natural, because days only couple through rebalancing costs.
  Try other cluster sizes, `HierarchicalStrategy` aggregation, or `hybrid.EnergyImpactDecomposer`.
* **Use the report.** Substitute `report.fixed_values` to shrink the QUBO before solving, and read
  `report.structure` to pick partition sizes.
* **Warm starts.** Seed QAOA from the service's angles. Use the best classical solution as
  `reference_states`. Restrict the search to the neighbourhood of an incumbent.
* **Hybrid loops.** Fix most variables to the incumbent, re-solve a window of assets × days with
  QAOA/PCE, accept if the official score improves, and repeat (large-neighbourhood search).
* **Scale out.** With `USE_CLOUD = True`, sub-programs run in parallel. Use more shots, more layers,
  `quantum_routine="iterative_qaoa"` or `"progressive_pce"`.

---
## Handing in

Keep every `.sol` file you write, with the instance, its size and the method and settings that produced
it. **Handing in** at the end of notebook 00 says how to submit.